# Sprint 3 - NqText Complete Test (Wikipedia - Easier Dataset)

**Model:** NVIDIA Nemotron-3 Ultra 550B

**Dataset:** NqText (Wikipedia factual Q&A)

**Why easier:** Short factual answers from Wikipedia (vs complex financial calculations)

**Total Q&A:** 71 questions

- Supreme Court of the United States: 57 questions ⭐ Most
- 2018 Tour de France: 13 questions
- Hannah John-Kamen: 1 question

**Metric:** Span-level F1 score (more lenient than Exact Match)

## Setup and Imports

In [ ]:
import pandas as pd
import chromadb
import PyPDF2
import time
import importlib.util
import os
from datetime import datetime
from together import Together
from langchain.text_splitter import RecursiveCharacterTextSplitter
import chromadb.utils.embedding_functions as embedding_functions
from uda.utils import preprocess, llm
from uda.eval.my_eval import eval_main

print("✓ All imports successful")

## Configuration

In [ ]:
# Load API config
_spec = importlib.util.spec_from_file_location(
    "access_config",
    os.path.join(os.getcwd(), "uda", "utils", "access_config.py")
)
access_config = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(access_config)

print(f"Model: {access_config.TOGETHER_MODEL}")
print(f"API Key: {access_config.TOGETHER_API_KEY[:20]}...")

In [ ]:
# Parameters
CHUNK_SIZE = 3000
CHUNK_OVERLAP = 300
TOP_K = 5
TEMPERATURE = 0.1
MAX_TOKENS = 512

# Documents to test (NqText - Wikipedia)
NQ_DOCS = [
    "Supreme Court of the United States",  # 57 Q&A - main document
    "2018 Tour de France",  # 13 Q&A
    "Hannah John-Kamen",  # 1 Q&A
]

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Chunk size: {CHUNK_SIZE}")
print(f"Top-K: {TOP_K}")
print(f"Documents: {len(NQ_DOCS)}")

## Initialize Models

In [ ]:
# Together AI client
together_client = Together(api_key=access_config.TOGETHER_API_KEY)
print("✓ Together AI client initialized")

# Embedding model (local, free)
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
print("✓ Embedding model loaded: all-MiniLM-L6-v2")

# Text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
print("✓ Text splitter initialized")

## Helper Functions

In [ ]:
def extract_pdf_text(pdf_path):
    """Extract text from PDF"""
    pdf_text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file, strict=False)
        for page_num in range(len(reader.pages)):
            pdf_text += reader.pages[page_num].extract_text()
    return pdf_text

def build_index(text_chunks, collection_name="temp_collection"):
    """Build vector index"""
    chroma_client = chromadb.Client()
    
    # Delete if exists
    try:
        chroma_client.delete_collection(collection_name)
    except:
        pass
    
    # Create collection
    collection = chroma_client.create_collection(
        collection_name,
        embedding_function=ef,
        metadata={"hnsw:space": "cosine"}
    )
    
    # Add documents
    id_list = [str(i) for i in range(len(text_chunks))]
    collection.add(documents=text_chunks, ids=id_list)
    
    return collection

def answer_question(collection, question):
    """Retrieve context and generate answer"""
    # Retrieve
    fetch_res = collection.query(query_texts=[question], n_results=TOP_K)
    context = "\n".join(fetch_res["documents"][0])
    
    # Build prompt (nq uses different template than fin)
    llm_message = llm.make_prompt(
        question=question,
        context=context,
        task_name="nq",
        llm_type="gpt-4"
    )
    
    # Generate
    response = together_client.chat.completions.create(
        model=access_config.TOGETHER_MODEL,
        messages=llm_message,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    
    return response.choices[0].message.content

print("✓ Helper functions defined")

## Load Q&A Data

In [ ]:
# Load all NqText Q&A
csv_file = "./dataset/qa/nq_qa.csv"
df = pd.read_csv(csv_file, sep="|", na_filter=False, dtype={"doc_name": str})
qas_dict = preprocess.qa_df_to_dict("nq", df)

# Count Q&A per document
total_qa = 0
for doc in NQ_DOCS:
    if doc in qas_dict:
        count = len(qas_dict[doc])
        total_qa += count
        print(f"{doc}: {count} Q&A pairs")

print(f"\nTotal: {total_qa} Q&A pairs")

## Main Processing Loop

**This will take approximately 25-30 minutes** (71 questions × ~20-25 seconds each)

**Expected performance:** 60-80% (easier than financial data)

In [ ]:
all_results = []

for doc_name in NQ_DOCS:
    print(f"\n{'='*80}")
    print(f"Processing: {doc_name}")
    print(f"{'='*80}")
    
    # Get PDF path
    pdf_path = preprocess.get_example_pdf_path("nq", doc_name)
    if not pdf_path:
        print(f"❌ PDF not found")
        continue
    
    print(f"PDF: {pdf_path}")
    
    # Extract and chunk
    print("Extracting text...")
    pdf_text = extract_pdf_text(pdf_path)
    text_chunks = text_splitter.split_text(pdf_text)
    print(f"Created {len(text_chunks)} chunks")
    
    # Build index
    print("Building vector index...")
    collection = build_index(text_chunks, collection_name=f"nq_{doc_name.replace(' ', '_')}")
    print("✓ Index built")
    
    # Get Q&A for this document
    doc_qas = qas_dict[doc_name]
    print(f"\nAnswering {len(doc_qas)} questions...")
    
    # Process each question
    for idx, qa in enumerate(doc_qas, 1):
        question = qa["question"]
        print(f"\n[{idx}/{len(doc_qas)}] {question[:70]}...")
        
        try:
            answer = answer_question(collection, question)
            print(f"   Answer: {answer[:80]}...")
            
            all_results.append({
                "question": question,
                "response": answer,
                "doc": doc_name,
                "q_uid": qa["q_uid"],
                "answers": qa["answers"],
                "dataset": "nq",
            })
            
            time.sleep(0.5)  # Rate limiting
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            continue
    
    print(f"\n✓ Completed {doc_name}: {len([r for r in all_results if r['doc'] == doc_name])} questions processed")

print(f"\n{'='*80}")
print(f"ALL DOCUMENTS PROCESSED")
print(f"{'='*80}")
print(f"Total Q&A processed: {len(all_results)}")

## Evaluate Results

In [ ]:
if all_results:
    print("\nEvaluating NqText results...")
    print("Metric: Span-level F1 score\n")
    eval_main("nq", all_results)
else:
    print("❌ No results to evaluate")

## Save Results

In [ ]:
if all_results:
    # Save to CSV
    results_df = pd.DataFrame(all_results)
    output_file = f"../nqtext_complete_results_{TIMESTAMP}.csv"
    results_df.to_csv(output_file, index=False)
    
    print(f"\n✓ Results saved to: {output_file}")
    print(f"Total Q&A: {len(results_df)}")
    
    # Summary by document
    print("\nResults by document:")
    for doc in NQ_DOCS:
        count = len(results_df[results_df['doc'] == doc])
        print(f"  {doc}: {count} questions")
else:
    print("❌ No results to save")

## Display Sample Results

In [ ]:
if all_results:
    # Show first 10 results
    print("\nSample Results (first 10):")
    for i, result in enumerate(all_results[:10], 1):
        print(f"\n{i}. Q: {result['question']}")
        print(f"   A: {result['response'][:100]}...")
        print(f"   Ground Truth: {result['answers']}")

## Summary Statistics & Comparison

In [ ]:
if all_results:
    results_df = pd.DataFrame(all_results)
    
    # Count empty responses
    results_df['response'] = results_df['response'].fillna('')
    empty_count = results_df['response'].str.strip().eq('').sum()
    answered_count = len(results_df) - empty_count
    
    print(f"\n{'='*80}")
    print(f"STATISTICS")
    print(f"{'='*80}")
    print(f"Total questions: {len(results_df)}")
    print(f"Answered: {answered_count} ({answered_count/len(results_df)*100:.1f}%)")
    print(f"Empty responses: {empty_count} ({empty_count/len(results_df)*100:.1f}%)")
    print(f"Avg response length: {results_df['response'].str.len().mean():.0f} characters")
    
    print(f"\n{'='*80}")
    print(f"COMPARISON WITH FINHYBRID")
    print(f"{'='*80}")
    print(f"\nFinHybrid (Financial - Hard):")
    print(f"  - 47 questions, ~20% accuracy, 40% empty")
    print(f"  - Metric: Exact Match (strict)")
    print(f"\nNqText (Wikipedia - Easier):")
    print(f"  - {len(results_df)} questions, F1 score shown above")
    print(f"  - Metric: Span F1 (lenient)")
    print(f"  - Empty rate: {empty_count/len(results_df)*100:.1f}%")
    print(f"\nExpected: NqText should score 2-3x higher than FinHybrid")

---

## Done!

**Results saved to:** `../nqtext_complete_results_[timestamp].csv`

**Next steps:**
1. Compare F1 score with FinHybrid's ~20% Exact Match
2. Analyze which types of questions Nemotron handles better
3. Decide if parameter optimization is worth it